<a href="https://colab.research.google.com/github/Riz2693/Eksperimen-TensorFlow/blob/main/Labelling_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install emoji

In [13]:
import pandas as pd
import numpy as np
import re
import string
import html
import emoji
import torch
import kagglehub
import os

from transformers import BertTokenizer, BertForSequenceClassification

In [3]:
path = kagglehub.dataset_download("khalidryder777/500k-chatgpt-tweets-jan-mar-2023")

Using Colab cache for faster access to the '500k-chatgpt-tweets-jan-mar-2023' dataset.


In [4]:
csv_name = 'Twitter Jan Mar.csv'
file_path = os.path.join(path, csv_name)
df = pd.read_csv(file_path)

df.head()

,date,id,content,username,like_count,retweet_count
0,2023-03-29 22:58:21+00:00,1641213230730051584,"Free AI marketing and automation tools, strate...",RealProfitPros,0.0,0.0
1,2023-03-29 22:58:18+00:00,1641213218520481805,@MecoleHardman4 Chat GPT says it’s 15. 😂,AmyLouWho321,0.0,0.0
2,2023-03-29 22:57:53+00:00,1641213115684536323,https://t.co/FjJSprt0te - Chat with any PDF!\n...,yjleon1976,0.0,0.0
3,2023-03-29 22:57:52+00:00,1641213110915571715,"AI muses: ""In the court of life, we must all f...",ChatGPT_Thinks,0.0,0.0
4,2023-03-29 22:57:26+00:00,1641213003260633088,Most people haven't heard of Chat GPT yet.\nFi...,nikocosmonaut,0.0,0.0


In [5]:
df = df[['content']]
df = df.head(500)

df.head()

,content
0,"Free AI marketing and automation tools, strate..."
1,@MecoleHardman4 Chat GPT says it’s 15. 😂
2,https://t.co/FjJSprt0te - Chat with any PDF!\n...
3,"AI muses: ""In the court of life, we must all f..."
4,Most people haven't heard of Chat GPT yet.\nFi...


In [6]:
def clean_data(text):
  if not isinstance(text, str):
    return ""
  text = html.unescape(text)
  text = re.sub(r"http\S+|www\S+|https\S+", "", text)
  text = re.sub(r"@\w+|#\w+", "", text)
  text = re.sub(r"\d+", "", text)
  text = emoji.replace_emoji(text, "")
  text = text.translate(str.maketrans("", "", string.punctuation))
  text = re.sub(r"\s+", " ", text).strip()
  text = text.lower()
  return text

In [7]:
df['clean_content'] = df['content'].apply(clean_data)

df.head()

,content,clean_content
0,"Free AI marketing and automation tools, strate...",free ai marketing and automation tools strateg...
1,@MecoleHardman4 Chat GPT says it’s 15. 😂,chat gpt says it’s
2,https://t.co/FjJSprt0te - Chat with any PDF!\n...,chat with any pdf check out how this new ai qu...
3,"AI muses: ""In the court of life, we must all f...",ai muses in the court of life we must all face...
4,Most people haven't heard of Chat GPT yet.\nFi...,most people havent heard of chat gpt yet first...


In [8]:
model_name = 'nlptown/bert-base-multilingual-uncased-sentiment'

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [15]:
labels = ["1 star", "2 stars", "3 stars", "4 stars", "5 stars"]

prediction = []
probs_list = []

for text in df['clean_content']:
  enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
  out = model(**enc)
  probs = torch.nn.functional.softmax(out.logits, dim=-1).detach().numpy()[0]
  pred = torch.argmax(torch.tensor(probs)).numpy()

  prediction.append(labels[pred])
  probs_list.append(probs)